In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
# Vertex AI Model Garden - Earth AI Imagery Models Deployment

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://github.com/google-research/remote-sensing/raw/refs/heads/main/remote_sensing/vertex_ai/notebooks/model_garden_remote_sensing_deployment.ipynb">
      <img alt="Workbench logo" src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" width="32px"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw%2Egithubusercontent%2Ecom%2Fgoogle%2Dresearch%2Fremote%2Dsensing%2Fmaster%2Fremote%5Fsensing%2Fvertex%5Fai%2Fnotebooks%2Fmodel%5Fgarden%5Fremote%5Fsensing%5Fdeployment%2Eipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/notebooks/model_garden_remote_sensing_deployment.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
</tr></tbody></table>

In [ ]:
# @title Initialize Vertex AI Remote Sensing model deployment

import sys
import os
from PIL import Image

!git clone https://github.com/google-research/remote-sensing.git
repo_path = '/content/remote-sensing'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from google.cloud import aiplatform
from remote_sensing.vertex_ai import utils

# Get the default cloud project id.
PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = "" # @param { 'type' : 'string' }
if not REGION:
    REGION = os.environ["GOOGLE_CLOUD_REGION"]
aiplatform.init(project=PROJECT_ID, location=REGION)

In [ ]:
# @title Create a model

# @markdown **Choose a name for the model (to be deployed)**
DISPLAY_NAME = "mammut-combined-test-l4"  # @param { 'type' : 'string' }
# @markdown **Specify the model type, variant mode and accelerator (platform) config.**
MODEL_TYPE = "MAMMUT"  # @param ["MAMMUT", "OWLVIT"]
MODEL_MODE = "COMBINED"  # @param ["IMAGE_ONLY", "TEXT_ONLY", "COMBINED"]
ACCELERATOR = "NVIDIA_L4"  # @param ["CPU", "NVIDIA_L4", "NVIDIA_A100_80GB"]

model: aiplatform.Model = utils.create_model(
    display_name=f"{DISPLAY_NAME}-model",
    model_type=MODEL_TYPE,
    model_mode=MODEL_MODE,
    accelerator=ACCELERATOR,
)

print(f"Created model {model.resource_name}")

In [ ]:
# @title Deploy the model to an Endpoint

# @markdown **Note:** For OWLVIT it is recommended to use a dedicated endpoint
# @markdown as it increases the input size from 1.5 MB to 10MB.
USE_DEDICATED_ENDPOINT = True  # @param { 'type' : 'boolean' }
# @markdown (optional) Specify the service account to use for deployment.
SERVICE_ACCOUNT = ""  # @param { 'type' : 'string' }
# @markdown **Specify the minimum and maximum number of replicas to use.**
MIN_REPLICA_COUNT = 1  # @param { 'type' : 'integer' }
MAX_REPLICA_COUNT = 1  # @param { 'type' : 'integer' }

endpoint: aiplatform.Endpoint = utils.deploy_model(
    endpoint_name=f"{DISPLAY_NAME}-endpoint",
    model=model,
    accelerator=ACCELERATOR,
    service_account=SERVICE_ACCOUNT,
    min_replica_count=MIN_REPLICA_COUNT,
    max_replica_count=MAX_REPLICA_COUNT,
    use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
)

## Inference examples

* Below there are 2 sets of samples: Object Detection (OWL-ViT) and  Classification (MaMMUT), make sure that the deployed endpoint has the correct model type, otherwise you can override it below.

* The samples are designed to work with the COMBINED mode, i.e. a variant
of the model that can accept text, image or both as input.

* Make sure you **cleanup unused resources** (endpoint) in the end. You can use
the cleanup section above.

* To get the best performance it is advised to use at least an **NVIDIA_L4 GPU**

In [ ]:
# @title Inference setup & utils.

# Download sample images
!wget -O harbor.jpg https://mrsg.aegean.gr/images/uploads/it2zi0eidej4ql33llj.jpg
!wget -O palace.jpeg https://www.spaceintelreport.com/wp-content/uploads/2021/05/Pleiades-NEO-US-Capitol-30cm.jpeg
harbor_img = Image.open("harbor.jpg")
palace_img = Image.open("palace.jpeg")

In [ ]:
# @markdown **(Optional)** Override the endpoint (use a different one).
# @markdown This is useful if you want to use a test a previously deployed model.
# @markdown otherwise the inference samples will use the recently deployed model.
ENDPOINT_ID = ""  # @param { 'type': 'string' }
use_dedicated_endpoint = True  # @param { 'type' : 'boolean' }

if ENDPOINT_ID:
  endpoint = aiplatform.Endpoint(ENDPOINT_ID)

In [ ]:
# @title Classification (MaMMUT) Inference Examples
# Make sure that the deployed endpoint above is a MaMMUT model.

# Call the image encoder with multiple images, batch_size is 1 by default.
result = endpoint.predict(
    instances=[
        {"image": utils.b64_png(harbor_img)},
        {"image": utils.b64_png(palace_img)},
    ],
    parameters={"batch_size": 2},
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print("Image encoder result, should contain 2 instances with embeddings.")
print(result)

# Call text encoder with multiple input instances
result = endpoint.predict(
    instances=[
        {"text": "text"},
        {"text": "second text"},
        {"text": "this is a longer sentence"},
        {"text": "this is a another long sentence, longer than the previous"},
    ],
    parameters={"batch_size": 2},
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print("Text encoder result, should contain 2 instances with embeddings.")
print(result)

# Call the zero-shot classification on the harbor & palace image, returns
# similarity scores for each image/text, used
labels = ["airport", "palace", "harbor", "shipyard", "park"]
result = endpoint.predict(
    instances=[
        {"image": utils.b64_png(harbor_img), "texts": labels},
        {"image": utils.b64_png(palace_img), "texts": labels},
    ],
    parameters={"batch_size": 2},
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print("Zero-shot classification result including similarity scores.")
print(result)

In [ ]:
# @title Object Detection (OWL-ViT) Inference Examples

# Make sure that the deployed endpoint above is OWL-ViT. It is advised to deploy
# a dedicated endpoint for OWL-ViT as the input size is relatively large.

# Call the image detection model, returns a list of object detections with
# bounding boxes, scores & embeddings.
result = endpoint.predict(
    instances=[
        {"image": utils.b64_png(harbor_img)},
    ],
    parameters={"batch_size": 1},
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print(
    "Image detection result, should contain 1 instance with object-level"
    " embeddings."
)
print(result)

# Call text encoder with multiple texts, returns text embeddings for each input.
result = endpoint.predict(
    instances=[
        {"text": "text"},
        {"text": "another text"},
        {"text": "this is a longer sentence"},
        {"text": "this is a very long sentence, even longer than above."},
    ],
    parameters={"batch_size": 4},
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print("Text encoder result, should contain 4 instances with text embeddings.")
print(result)

# Call the Open Vocabulary Detection mode with image/texts pairs, returns
# object detections and labels, including bounding boxes, scores & embeddings.
labels = ["ship", "harbor", "dome", "building", "bridge"]
result = endpoint.predict(
    instances=[
        {"image": utils.b64_png(harbor_img), "texts": labels},
        {"image": utils.b64_png(palace_img), "texts": labels},
    ],
    parameters={
        "batch_size": 4,
        # Return only the top 100 detections based on objectness_score.
        "top_k_objects": 100,
        # Discard the object/text embeddings, overall reduces the output size.
        "keep_embeddings": False,
    },
    use_dedicated_endpoint=use_dedicated_endpoint,
)
print(
    "Object detection result, including detection results with 100 objects"
    " each."
)
print(result)

In [ ]:
# @title Cleanup Resources
# @markdown  Delete the experiment models and endpoints to recycle the resources
# @markdown  and avoid unnecessary continuous charges that may incur.

endpoint.delete(force=True)
model.delete()